In [1]:
import os
import json
import copy
import requests
import pandas as pd
import numpy as np
import pickle as pkl
from tqdm import tqdm
from random import sample, choice, shuffle
import scipy
from scipy import sparse
from scipy.sparse import csr_matrix
from sklearn.linear_model import SGDClassifier
# from custom_models.custom_model import BatchVectorizerClassifierCitations
import xgboost as xgb
from sklearn.utils.multiclass import unique_labels
from sklearn.feature_extraction.text import HashingVectorizer
from sklearn import metrics
from sklearn.decomposition import PCA, IncrementalPCA
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.neighbors import KNeighborsClassifier
import importlib

import open_alex_utils

importlib.reload(open_alex_utils)

from open_alex_utils import get_pos_citation_dataset, \
    get_neg_citation_dataset, extract_bow_feats, get_bow_inputs_labels

/home/k2141255/anaconda3/envs/study_design_classifier/lib/python3.8/site-packages/xgboost/compat.py:36: FutureWarning: pandas.Int64Index is deprecated and will be removed from pandas in a future version. Use pandas.Index with the appropriate dtype instead.
  from pandas import MultiIndex, Int64Index


In [2]:
file_list = [file for file in os.listdir('open_alex_citations') if 'open' in file]

In [3]:
ids_file = '/home/k2141255/study_design_classifier/data/cleaned_pos_neg_train_val_test_dataset_ids.pkl'

In [4]:
study_design = 'is_guideline'

In [5]:
consort_pmids = [
    20335313,
    20334633,
    20332509,
    20346629,
    20410783,
    21686296,
    20352064,
    20334632,
]

In [6]:
consort_openalex_ids = []
for pmid in tqdm(consort_pmids):
    response = requests.get(f"http://api.openalex.org/works/pmid:{pmid}")
    consort_openalex_ids.append(response.json()['id'])

  0%|                                                                | 0/8 [00:00<?, ?it/s]


JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [ ]:
consort_openalex_ids

In [9]:
PMIDS = '../data/pmids.pkl'
with open(PMIDS, 'rb') as f:
    pmids = pkl.load(f)

In [10]:
pmids.keys()

dict_keys(['is_rct', 'is_sr', 'is_casecontrol', 'is_cohort', 'is_guideline'])

In [11]:
type(response.status_code)

int

In [12]:
pos_train_set, pos_train_feat_lst = get_pos_citation_dataset(pmids[study_design]['pos']['train'])
pos_val_set, pos_val_feat_lst = get_pos_citation_dataset(pmids[study_design]['pos']['val'])
pos_test_set, pos_test_feat_lst = get_pos_citation_dataset(pmids[study_design]['pos']['test'])

100%|████████████████████████████████████████████████| 4000/4000 [00:00<00:00, 8578.44it/s]


In [13]:
neg_train_set, neg_train_feat_lst = get_neg_citation_dataset(pmids[study_design]['neg']['train'], len(pos_train_set))
neg_val_set, neg_val_feat_lst = get_neg_citation_dataset(pmids[study_design]['neg']['val'])
neg_test_set, neg_test_feat_lst = get_neg_citation_dataset(pmids[study_design]['neg']['test'])

100%|████████████████████████████████████████████████| 4000/4000 [00:03<00:00, 1262.99it/s]


In [14]:
feat_lst = pos_train_feat_lst + pos_val_feat_lst + pos_test_feat_lst + neg_train_feat_lst + neg_val_feat_lst + neg_test_feat_lst 

In [15]:
len(neg_train_set)

3776

In [16]:
len(pos_train_set)

3776

In [17]:
feat_lst = list(set(feat_lst))

In [18]:
len(feat_lst)

661024

In [19]:
from collections import defaultdict

freqs = defaultdict(int)

for paper in tqdm(pos_train_set):
    for opid in paper:
        freqs[opid] +=1

for paper in tqdm(neg_train_set):
    for opid in paper:
        freqs[opid] +=1

100%|██████████████████████████████████████████████| 3776/3776 [00:00<00:00, 267745.67it/s]


In [20]:
freqs = {k: v for k, v in sorted(freqs.items(), key=lambda item: item[1], reverse=True)}

In [21]:
freqs

{'meta': 670,
 'results': 670,
 'group_by': 670,
 'https://openalex.org/W2165010366': 107,
 'https://openalex.org/W1247968195': 51,
 'https://openalex.org/W2104675861': 39,
 'https://openalex.org/W1775749144': 36,
 'https://openalex.org/W2093770528': 32,
 'https://openalex.org/W2100837269': 30,
 'https://openalex.org/W2135400982': 26,
 'https://openalex.org/W2129688566': 25,
 'https://openalex.org/W2010149513': 24,
 'https://openalex.org/W2156121263': 22,
 'https://openalex.org/W1994277369': 21,
 'https://openalex.org/W1980717583': 19,
 'https://openalex.org/W2067661817': 19,
 'https://openalex.org/W2096074908': 19,
 'https://openalex.org/W2134236426': 19,
 'https://openalex.org/W2079065919': 18,
 'https://openalex.org/W1847168837': 17,
 'https://openalex.org/W2119790985': 16,
 'https://openalex.org/W2337454357': 16,
 'https://openalex.org/W1994158325': 16,
 'https://openalex.org/W311320695': 16,
 'https://openalex.org/W2473559982': 15,
 'https://openalex.org/W2588681363': 15,
 'https:

In [43]:
freqs_reduced = {k: v for k, v in freqs.items() if  v > 3}

In [44]:
len(freqs_reduced)

1577

In [45]:
feature_lst_reduced = list(freqs_reduced)

In [46]:
lim_len = -1

In [47]:
bow_neg_train = extract_bow_feats(neg_train_set, feature_lst_reduced)
bow_neg_val = extract_bow_feats(neg_val_set, feature_lst_reduced)
bow_neg_test = extract_bow_feats(neg_test_set, feature_lst_reduced)
bow_pos_train = extract_bow_feats(pos_train_set, feature_lst_reduced)
bow_pos_val = extract_bow_feats(pos_val_set, feature_lst_reduced)
bow_pos_test = extract_bow_feats(pos_test_set, feature_lst_reduced)

100%|████████████████████████████████████████████████| 3775/3775 [00:02<00:00, 1729.07it/s]


In [48]:
inputs_train, labels_train = get_bow_inputs_labels(bow_pos_train, bow_neg_train, is_shuffle = True)
inputs_val, labels_val = get_bow_inputs_labels(bow_pos_val, bow_neg_val, is_shuffle = True)
inputs_test, labels_test = get_bow_inputs_labels(bow_pos_test, bow_neg_test, is_shuffle = True)

100%|███████████████████████████████████████████████████| 77/77 [00:00<00:00, 34390.52it/s]


In [49]:
len(labels_train)

7550

In [50]:
len(inputs_train)

7550

In [51]:
class BatchVectorizerClassifierCitations:
    """Custom vectorizer and classifier class to process data in
    batches.  Useful when there are RAM and CPU limitations.

    Args:
        BaseEstimator (Estimator): base class for estimator
    """

    def __init__(
        self,
        loss="hinge",
        penalty="l2",
        alpha=0.0001,
        l1_ratio=0.15,
        fit_intercept=True,
        max_iter=1000,
        tol=0.001,
        shuffle=True,
        verbose=0,
        epsilon=0.1,
        n_jobs=None,
        random_state=None,
        learning_rate="optimal",
        eta0=0.0,
        power_t=0.5,
        early_stopping=False,
        validation_fraction=0.1,
        n_iter_no_change=5,
        class_weight=None,
        warm_start=False,
        average=False,
    ):
        """intialization of class

        Args:
            clf (Estimator): classifier for training
            vectorizer (Vectorizer): vectorizer to turn text into vectors
            batch_size (int): size of each batch used for training
        """

        self.clf = SGDClassifier(
            loss=loss,
            penalty=penalty,
            alpha=alpha,
            l1_ratio=l1_ratio,
            fit_intercept=fit_intercept,
            max_iter=max_iter,
            tol=tol,
            shuffle=shuffle,
            verbose=verbose,
            epsilon=epsilon,
            n_jobs=n_jobs,
            random_state=random_state,
            learning_rate=learning_rate,
            eta0=eta0,
            power_t=power_t,
            early_stopping=early_stopping,
            validation_fraction=validation_fraction,
            n_iter_no_change=n_iter_no_change,
            class_weight=class_weight,
            warm_start=warm_start,
            average=average,
        )
        self.vectorizer = HashingVectorizer(ngram_range=(1, 4), stop_words="english")
        self.batch_size = 1000
        # self._estimator_type = "classifier"

    def fit(self, X, y):
        """fit the vectorizer and the classifier

        Args:
            X (pd.Series(str)): inputs
            y (np.array(int)): labels
        """

        # check whether vectorizer has been fitted
        vec_fitted = False

        # number of batches to be processed
        if len(y) >= self.batch_size:
            num_batches = len(y) // self.batch_size
        else:
            num_batches = 1

        # Store the classes seen during fit
        self.classes_ = unique_labels(y)

        # for X_chunk, y_chunk in tqdm(
        #     zip(np.array_split(X, num_batches), np.array_split(y, num_batches),),
        #     total=num_batches,
        # ):

        for X_chunk, y_chunk in zip(
            np.array_split(X, num_batches), np.array_split(y, num_batches)
        ):
            self.clf.fit(X_chunk, y_chunk)

        return self

    def predict(self, X):

#         X_unicode = X.values.astype("U")
#         X_trans = self.vectorizer.transform(X_unicode)

        predictions = self.clf.predict(X)
        return predictions

    # def predict_proba(self, X):
    #     # Check if fit has been called
    #     check_is_fitted(self)

    #     # Input validation
    #     X = check_array(X)
    #     X_unicode = X.values.astype("U")
    #     X_trans = self.vectorizer.transform(X_unicode)

    #     proba = self.clf.predict_proba(X_trans)
    #     return proba

    def get_params(self, deep):
        return self.clf.get_params(deep=deep)

    def set_params(self, **parameters):
        return self.clf.set_params(**parameters)

In [52]:
clf = BatchVectorizerClassifierCitations()
clf.fit(inputs_train, labels_train)

In [32]:
preds_train = clf.predict(inputs_train)
preds_val = clf.predict(inputs_val)
preds_test = clf.predict(inputs_test)

In [40]:
douts = clf.clf.decision_function(inputs_test)
preds_test_mod = [1 if dout>-1 else 0 for dout in douts]

In [41]:
def print_metrics(labels, preds):
    print("recall:", metrics.recall_score(labels, preds))
    print("precision:", metrics.precision_score(labels, preds))
    print("f1:",  metrics.f1_score(labels, preds))
    print("accuracy", metrics.accuracy_score(labels, preds))

In [42]:
print_metrics(labels_train, preds_train)

recall: 0.20603110983872508
precision: 0.904861693210394
f1: 0.33563933151962694
accuracy 0.5921843687374749


In [43]:
print_metrics(labels_val, preds_val)

recall: 0.17754569190600522
precision: 0.03494946033921535
f1: 0.05840251932436301
accuracy 0.9733035714285714


In [44]:
print_metrics(labels_test, preds_test)

recall: 0.022837067927520483
precision: 0.5
f1: 0.043679129366333155
accuracy 0.5


In [45]:
print_metrics(labels_test, preds_test_mod)

recall: 0.9458679627617398
precision: 0.5
f1: 0.6541869569853707
accuracy 0.5
